# Métricas Climáticas Temporais: Amplitude Térmica Diurna e Graus-Hora

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/06_temporal_climate_metrics.pt.ipynb)

**Padrões de plotagem.** Toda função que gera figura no LCZ4py agora retorna por padrão um `Figure` estático nativo do matplotlib (300 dpi, pronto para publicação, sem navegador nem Kaleido). Passe `renderer="plotly"` a qualquer função de plotagem para obter a versão interativa.

**Objetivo de aprendizagem**: aprender a calcular dois indicadores climáticos
temporais clássicos — Amplitude Térmica Diurna (DTR) e Graus-Hora de
Refrigeração/Aquecimento (CDH/HDH) — a partir de registros de temperatura do ar
de estações, e a agregar ambos por classe de Zona Climática Local (LCZ) usando
`lcz_dtr` e `lcz_degree_hours` do **LCZ4py**. Usamos o conjunto de dados de
estações de Berlim (`lcz_data_berlin.csv`) ao longo de todo o notebook,
consistente com o restante da série `local/`.

## Por que DTR e Graus-Hora importam para o clima urbano

A **Amplitude Térmica Diurna (DTR)** — a diferença entre a temperatura máxima e
mínima diárias do ar em uma estação — é uma das assinaturas mais antigas e
robustas da urbanização no registro climático. O esquema de Zonas Climáticas
Locais de Stewart & Oke (Stewart & Oke, 2012, *Bulletin of the American
Meteorological Society*) foi criado justamente porque o binômio "urbano vs.
rural" é grosseiro demais para explicar por que alguns sítios construídos
aquecem à noite enquanto outros não: núcleos compactos de edifícios altos
(LCZ 1–3), subúrbios abertos de baixa altura (LCZ 6) e zonas industriais
(LCZ 8) têm fatores de visão do céu, admitância térmica e liberação de calor
antropogênico muito diferentes entre si, e portanto ciclos diurnos de
temperatura muito distintos mesmo sendo todos "urbanos".

Mecanisticamente, LCZs construídas e densas tendem a **suprimir a DTR** em
relação a LCZs vegetadas ou dominadas por água (11–17): materiais de
construção de alta admitância térmica (concreto, asfalto) absorvem e
armazenam radiação solar durante o dia, e um baixo fator de visão do céu
(edifícios altos, cânions urbanos) retém a radiação de onda longa emitida à
noite, retardando o resfriamento radiativo. O efeito líquido é uma mínima
diária atenuada — as máximas diurnas são semelhantes ou só ligeiramente
maiores que em sítios rurais, mas as mínimas noturnas permanecem
marcadamente mais quentes. Essa assimetria (pequeno aquecimento diurno,
grande aquecimento noturno) é uma das explicações mecanísticas clássicas
para a **Ilha de Calor Urbana (UHI) noturna**, e comparar a DTR entre
classes LCZ com a *mesma* rede de estações é um teste direto e orientado a
dados desse mecanismo — sem precisar de rótulos separados de "urbano" e
"rural", apenas a classe LCZ em que cada estação se encontra.

**Graus-Hora** — Graus-Hora de Refrigeração (CDH) e de Aquecimento (HDH) —
são a proxy padrão de demanda energética usada em simulação de energia de
edificações, previsão de carga de concessionárias e estudos de exposição ao
calor em saúde pública. Em relação a uma temperatura de conforto/balanço
(`base_temp`, comumente 18°C na prática europeia e americana), o CDH acumula
o quanto e por quanto tempo a temperatura do ar excede esse limiar (demanda
de refrigeração/ar-condicionado e risco de exposição ao calor), enquanto o
HDH acumula o quanto e por quanto tempo ela fica abaixo dele (demanda de
aquecimento). Como ambos são calculados **por hora** e depois somados por
dia, capturam tanto a *intensidade* quanto a *duração* do estresse térmico —
algo que uma simples média diária de temperatura não consegue.

Estratificar CDH/HDH por classe LCZ transforma essa proxy de demanda
energética em um diagnóstico de forma urbana: se LCZs compactas/impermeáveis
(1–3, 8–10) apresentam CDH sistematicamente maior que LCZs abertas/vegetadas
(6, 9, 11–17) sob as mesmas condições ambientais, isso é evidência direta de
que a forma urbana — não apenas o clima regional — impulsiona a carga
energética de refrigeração e a exposição ao calor. Esse é exatamente o tipo
de evidência que alimenta o planejamento de adaptação climática e de ações
de calor: identifica quais classes LCZ são alvos prioritários para
retrofits de telhados frios, arborização urbana ou intervenções de
sombreamento. Veja Anjos et al. (2025, *Scientific Reports*,
https://www.nature.com/articles/s41598-025-92000-0) para a metodologia
LCZ4r que este port em Python (LCZ4py) implementa, e Stewart & Oke (2012)
para o esquema original de classificação LCZ.

## Instalar o LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Configuração: mapa LCZ e dados de estações

Reutilizamos o mesmo mapa LCZ de Berlim e o mesmo CSV de estações do restante
da série `local/`: `lcz_get_map(city="Berlin")` para o raster LCZ, e
`lcz_data_berlin.csv` para as observações horárias de temperatura do ar
(`var="airT"`, `station_id="station"`).

In [ ]:
import pandas as pd
from IPython.display import display
from LCZ4py import lcz_get_map
from LCZ4py.local.lcz_temporal import lcz_dtr, lcz_degree_hours

lcz_map_path = lcz_get_map(city="Berlin")
df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df.head()

## `lcz_dtr` — Amplitude Térmica Diurna por classe LCZ

`lcz_dtr(x, data_frame, var, station_id, sp_res=100.0, tp_res="day", by=None, lang="en", year=None, month=None, day=None, hour=None, start=None, end=None)`
calcula, para **cada estação e cada dia calendário**, `DTR = max(T) − min(T)`
a partir das observações horárias (ou sub-diárias), e então atribui cada
estação à sua classe LCZ via o raster LCZ `x` (busca no pixel mais próximo
à latitude/longitude da estação).

Principais parâmetros:
- `x` — o raster LCZ (caminho ou dataset `rasterio`), usado apenas para
  classificar as estações em LCZ 1–17.
- `data_frame`, `var`, `station_id` — a tabela de observações e as colunas
  que contêm a variável de temperatura e o identificador da estação.
- `sp_res` — resolução espacial em metros; informativa aqui (nenhuma
  interpolação em grade é realizada, logo não afeta o resultado).
- `tp_res` — resolução temporal, atualmente `"day"` (DTR é intrinsecamente
  uma quantidade diária).
- `by` — agrupamento temporal opcional (ex.: `"season"`) para facetamento.
- `year`/`month`/`day`/`hour`/`start`/`end` — os filtros usuais de
  data/hora compartilhados entre as funções `local/`.

Retorna um dataclass `DTRResult` com:
- `df` — um DataFrame Polars, uma linha por estação por dia, colunas
  `station, lcz, date, dtr, t_max, t_min, lcz_name`.
- `plot` — um box plot estático do matplotlib das distribuições de DTR
  agrupadas por classe LCZ (renderizador padrão; passe `renderer="plotly"`
  para a versão interativa do Plotly), com uma anotação separando urbano
  (LCZ 1–10) de classes naturais/água (LCZ 11–17).

**O que observar**: se as caixas das LCZs urbanas (lado esquerdo) estiverem
sistematicamente *mais baixas* que as das LCZs naturais (lado direito), essa
é a assinatura de DTR suprimida descrita acima — evidência de um mecanismo
de UHI noturna operando nesta rede de estações.

In [ ]:
dtr_result = lcz_dtr(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    lang="pt",
)
dtr_result.df.head()

In [ ]:
display(dtr_result.plot)  # figura matplotlib estática (padrão)


### Interpretando a saída da DTR

Observe a mediana e o intervalo interquartil de cada caixa por LCZ: classes
compactas/construídas tipicamente se agrupam em DTR menor do que classes
abertas, de baixa altura ou naturais. Caixas largas (IQR grande) para uma
dada LCZ sugerem forte variabilidade dia a dia — por exemplo, impulsionada
por cobertura de nuvens ou vento — enquanto uma diferença entre medianas de
LCZ para LCZ é o sinal estrutural, causado pela forma urbana, que buscamos.
A tabela `df` permite investigar dias-estação individuais (por exemplo, para
encontrar os dias com o maior contraste urbano-rural de DTR).

## `lcz_degree_hours` — Graus-Hora de Refrigeração/Aquecimento por classe LCZ

`lcz_degree_hours(x, data_frame, var, station_id, base_temp=18.0, degree_type="cooling", sp_res=100.0, by=None, lang="en", year=None, month=None, day=None, hour=None, start=None, end=None)`
calcula graus-hora horários em relação a `base_temp` e os acumula por
estação por dia:

- **Refrigeração** (`degree_type="cooling"`): `CDH = Σ max(0, T − base_temp)`
  por hora, somado por dia — proxy de demanda de ar-condicionado / exposição
  ao calor.
- **Aquecimento** (`degree_type="heating"`): `HDH = Σ max(0, base_temp − T)`
  por hora, somado por dia — proxy de demanda de aquecimento de ambientes.

Principais parâmetros:
- `base_temp` — o limiar de conforto/balanço em °C (padrão 18,0, convenção
  comum europeia/americana).
- `degree_type` — `"cooling"` ou `"heating"`.
- Os demais parâmetros (`x`, `data_frame`, `var`, `station_id`, `sp_res`,
  `by`, filtros de data) espelham `lcz_dtr`.

Retorna um dataclass `DegreeHoursResult` com:
- `df` — graus-hora acumulados diários por estação, colunas
  `station, lcz, date, degree_hours, lcz_name`.
- `total` — agregado por classe LCZ, colunas `lcz, lcz_name, mean, std, total`.
- `plot` — um gráfico de barras estático do matplotlib (renderizador padrão;
  passe `renderer="plotly"` para a versão interativa do Plotly) do total de
  graus-hora por classe LCZ (barras de erro = desvio padrão entre
  dias-estação), também anotado urbano vs. natural.

Executamos a função duas vezes abaixo — uma para refrigeração, outra para
aquecimento — para observar as duas pontas do espectro de conforto térmico
na mesma rede de estações.

In [ ]:
cdh_result = lcz_degree_hours(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    base_temp=18.0,
    degree_type="cooling",
    lang="pt",
)
cdh_result.total

In [ ]:
display(cdh_result.plot)  # figura matplotlib estática (padrão)


In [ ]:
hdh_result = lcz_degree_hours(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    base_temp=18.0,
    degree_type="heating",
    lang="pt",
)
hdh_result.total

In [ ]:
display(hdh_result.plot)  # figura matplotlib estática (padrão)


### Interpretando a saída de Graus-Hora

Na execução de refrigeração, as classes LCZ com as barras mais altas são as
que acumulam mais horas acima do limiar de conforto — são os alvos
prioritários para redução de demanda de energia de refrigeração
(sombreamento, telhados frios, arborização) e para ações de alerta precoce
de saúde relacionadas ao calor. Na execução de aquecimento, o padrão
frequentemente (embora nem sempre) se inverte: classes abertas/vegetadas ou
de maior elevação podem acumular mais graus-hora de aquecimento se
apresentarem temperaturas médias mais baixas. Comparar as duas lado a lado
diz a um planejador de adaptação climática quais classes LCZ são
"dominadas por refrigeração" (urbano compacto), quais são "dominadas por
aquecimento" e quais são equilibradas — informando diretamente onde
priorizar investimento em adaptação. A tabela `total` é o resumo compacto
para exportar em um relatório; `df` fornece a série diária completa por
estação para trabalho mais aprofundado de série temporal (ex.: combinando
com `lcz_ts` do notebook 1).

## Conclusão e próximos passos

Neste notebook calculamos duas métricas climáticas temporais — Amplitude
Térmica Diurna e Graus-Hora de Refrigeração/Aquecimento — a partir da rede
de estações de Berlim, agregando ambas por classe LCZ com `lcz_dtr` e
`lcz_degree_hours`. A DTR nos deu uma janela mecanística para a UHI noturna
(amplitude diurna suprimida em LCZs urbanas compactas), enquanto o CDH/HDH
traduziu temperatura em uma proxy de demanda energética e exposição ao
calor, estratificada por forma urbana.

**Próximo**: [`07_thermal_comfort_anthropogenic_heat`](07_thermal_comfort_anthropogenic_heat.pt.ipynb)
constrói diretamente sobre esta base — calcula o Índice Climático Térmico
Universal (UTCI) para avaliação de estresse térmico humano e estima o
fluxo de calor antropogênico (Q_F) por classe LCZ, estendendo a história de
balanço energético iniciada aqui com os graus-hora.

## Gráficos de métricas prontos para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e as cores para que gráficos de DTR e graus-hora possam ser exportados como figuras prontas para manuscrito com um argumento extra.

Um exemplo curto pronto para publicação:

- `style="science"` para o gráfico da métrica.
- `isave=True` com `save_extension="png"` ou `"pdf"` para gravar a figura final.


In [ ]:
dtr_pub = lcz_dtr(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    style="science",
    isave=True,
    save_extension="png",
    lang="pt",
)
display(dtr_pub.plot)  # figura matplotlib estática (padrão)

dh_pub = lcz_degree_hours(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    degree_type="cooling",
    style="science",
    isave=True,
    save_extension="pdf",
    lang="pt",
)
display(dh_pub.plot)  # figura matplotlib estática (padrão)
